In [38]:
pip install scikit-learn gensim


Note: you may need to restart the kernel to use updated packages.


In [40]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import gensim.downloader as api
import numpy as np

In [41]:
# --------------------------------------------------
# Step 1: Corpus and Query
# --------------------------------------------------

corpus = [
    "Machine learning enables computers to learn from data",
    "Deep learning is a subset of machine learning",
    "Large language models use tokenization techniques",
    "Artificial intelligence is transforming industries",
    "Transformers are widely used in natural language processing"
]
query = "AI and deep learning are related fields"

print("Corpus:")
for c in corpus:
    print("-", c)
print("\nQuery:", query)

Corpus:
- Machine learning enables computers to learn from data
- Deep learning is a subset of machine learning
- Large language models use tokenization techniques
- Artificial intelligence is transforming industries
- Transformers are widely used in natural language processing

Query: AI and deep learning are related fields


In [42]:
def best_match(scores):
    return corpus[np.argmax(scores)]


def sent_vector(sentence, model, is_glove=False):
    """Average the word vectors of a sentence into one vector."""
    words = sentence.lower().split()
    if is_glove:
        vecs = [model[w] for w in words if w in model]
    else:
        vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(50)


In [43]:
# --------------------------------------------------
# Step 2: Bag of Words
# --------------------------------------------------

print("\n===== Bag-of-Words =====")

bow = CountVectorizer()
bow_matrix = bow.fit_transform(corpus + [query])

print("Embedding for query (first 10 values):")
print(bow_matrix.toarray()[-1][:10])

bow_scores = cosine_similarity(bow_matrix[-1], bow_matrix[:-1])[0]
for i, s in enumerate(bow_scores):
    print(f"Similarity with sentence {i+1}: {s:.3f}")
print("Best match:", best_match(bow_scores))



===== Bag-of-Words =====
Embedding for query (first 10 values):
[1 1 1 0 0 0 1 0 1 0]
Similarity with sentence 1: 0.134
Similarity with sentence 2: 0.378
Similarity with sentence 3: 0.000
Similarity with sentence 4: 0.000
Similarity with sentence 5: 0.134
Best match: Deep learning is a subset of machine learning


In [44]:
# --------------------------------------------------
# Step 3: TF-IDF
# --------------------------------------------------

print("\n===== TF-IDF =====")

tfidf = TfidfVectorizer()
matrix = tfidf.fit_transform(corpus + [query])

print("Embedding for query (first 10 values):")
print(matrix[-1].toarray()[0][:10])

tfidf_scores = cosine_similarity(matrix[-1], matrix[:-1])[0]
for i, s in enumerate(tfidf_scores):
    print(f"Similarity with sentence {i+1}: {s:.3f}")
print("Best match:", best_match(tfidf_scores))



===== TF-IDF =====
Embedding for query (first 10 values):
[0.41436586 0.41436586 0.33978594 0.         0.         0.
 0.33978594 0.         0.41436586 0.        ]
Similarity with sentence 1: 0.074
Similarity with sentence 2: 0.277
Similarity with sentence 3: 0.000
Similarity with sentence 4: 0.000
Similarity with sentence 5: 0.103
Best match: Deep learning is a subset of machine learning


In [45]:
# --------------------------------------------------
# Step 4: Word2Vec (trained on our own corpus)
# --------------------------------------------------

print("\n===== Word2Vec =====")

tokens = [s.lower().split() for s in corpus]
w2v = Word2Vec(sentences=tokens, vector_size=50, window=5, min_count=1, workers=1)

w2v_query = sent_vector(query, w2v)
print("Embedding for query (first 10 values):")
print(w2v_query[:10])

w2v_vecs = [sent_vector(s, w2v) for s in corpus]
w2v_scores = cosine_similarity([w2v_query], w2v_vecs)[0]
for i, s in enumerate(w2v_scores):
    print(f"Similarity with sentence {i+1}: {s:.3f}")
print("Best match:", best_match(w2v_scores))



===== Word2Vec =====
Embedding for query (first 10 values):
[ 0.00411972 -0.00050637  0.0144358   0.00602506 -0.01183152 -0.00983915
 -0.00160067  0.00376436 -0.00516782 -0.00871716]
Similarity with sentence 1: 0.348
Similarity with sentence 2: 0.595
Similarity with sentence 3: 0.001
Similarity with sentence 4: 0.342
Similarity with sentence 5: 0.205
Best match: Deep learning is a subset of machine learning


In [46]:
# --------------------------------------------------
# Step 5: GloVe (pretrained vectors)
# --------------------------------------------------

print("\n===== GloVe =====")
print("Downloading pretrained GloVe vectors (needs internet, first run only)...")

glove = api.load("glove-wiki-gigaword-50")  # 50-dim pretrained vectors

glove_query = sent_vector(query, glove, is_glove=True)
print("Embedding for query (first 10 values):")
print(glove_query[:10])

glove_vecs = [sent_vector(s, glove, is_glove=True) for s in corpus]
glove_scores = cosine_similarity([glove_query], glove_vecs)[0]
for i, s in enumerate(glove_scores):
    print(f"Similarity with sentence {i+1}: {s:.3f}")
print("Best match:", best_match(glove_scores))



===== GloVe =====
Embedding for query (first 10 values):
[ 0.28538314  0.36184087 -0.07564986  0.23678243  0.01277103  0.31524143
 -0.1645423  -0.37960857 -0.12307441  0.001475  ]
Similarity with sentence 1: 0.857
Similarity with sentence 2: 0.899
Similarity with sentence 3: 0.810
Similarity with sentence 4: 0.852
Similarity with sentence 5: 0.885
Best match: Deep learning is a subset of machine learning


In [36]:
# --------------------------------------------------
# Step 6: Final Comparison
# --------------------------------------------------

print("\n===== Best Match by Technique =====")
print("Bag-of-Words:", best_match(bow_scores))
print("TF-IDF      :", best_match(tfidf_scores))
print("Word2Vec    :", best_match(w2v_scores))
print("GloVe       :", best_match(glove_scores))



===== Best Match by Technique =====
Bag-of-Words: Deep learning is a subset of machine learning
TF-IDF      : Deep learning is a subset of machine learning
Word2Vec    : Deep learning is a subset of machine learning
GloVe       : Deep learning is a subset of machine learning
